In [1]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!nvcc --version | tail -n 2

name, compute_cap
Tesla T4, 7.5
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


## Install MiniKernel

If you uploaded `minikernel.zip` to this notebook, run the next cell.
If you cloned from GitHub, skip unzip and set `ROOT` to the repo folder.

In [2]:
import os
import sys
from pathlib import Path

if Path("/content/minikernel.zip").exists():
    !unzip -qo /content/minikernel.zip -d /content

candidates = [
    Path("/content/minikernel"),
    Path("/content/NvidiaPrep/minikernel"),
    Path.cwd() / "minikernel",
    Path.cwd(),
]
ROOT = next(p for p in candidates if (p / "pyproject.toml").exists() and (p / "examples/add_one.mk").exists())
os.chdir(ROOT)
print("project root:", ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
for name in list(sys.modules):
    if name == "minikernel" or name.startswith("minikernel."):
        del sys.modules[name]
!pip install -q -e .
from minikernel import frontend
print("imported", frontend.__file__)
try:
    import cupy
    print("cupy ok", cupy.__version__)
except ImportError:
    !pip install -q cupy-cuda12x
    import cupy
    print("cupy installed", cupy.__version__)

project root: /content/minikernel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for minikernel (pyproject.toml) ... done
imported /content/minikernel/src/minikernel/frontend.py
cupy ok 14.0.1


In [3]:
from pathlib import Path
import numpy as np
import cupy as cp

from minikernel.frontend import compile_file
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.interpreter import interpret

kernel = compile_file("examples/add_one.mk")
ptx = compile_to_ptx(kernel)
Path("add_one.ptx").write_text(ptx)
print(ptx)

.version 7.0
.target sm_75
.address_size 64

.visible .entry add_one(
    .param .u64 a_ptr,
    .param .u64 c_ptr,
    .param .u32 n_param
)
{
    .reg .pred %p<3>;
    .reg .b32 %r<9>;
    .reg .b64 %rd<9>;

    ld.param.u64 %rd1, [a_ptr];
    cvta.to.global.u64 %rd2, %rd1;
    ld.param.u64 %rd3, [c_ptr];
    cvta.to.global.u64 %rd4, %rd3;
    ld.param.u32 %r1, [n_param];
    mov.u32 %r2, %ctaid.x;
    mov.u32 %r3, %ntid.x;
    mov.u32 %r4, %tid.x;
    mad.lo.s32 %r5, %r2, %r3, %r4;
    setp.ge.u32 %p1, %r5, %r1;
    @%p1 bra L_done;
    setp.lt.u32 %p2, %r5, %r1;
    mul.wide.s32 %rd5, %r5, 4;
    add.s64 %rd6, %rd2, %rd5;
    mov.u32 %r6, 0;
    @%p2 ld.global.u32 %r6, [%rd6];
    mov.s32 %r7, 1;
    add.s32 %r8, %r6, %r7;
    mul.wide.u32 %rd7, %r5, 4;
    add.s64 %rd8, %rd4, %rd7;
    st.global.u32 [%rd8], %r8;
L_done:
    ret;
}



In [4]:
n = 8
host_in = np.arange(n, dtype=np.int32) * 10  # 0 10 20 30 40 50 60 70
ref = interpret(kernel, {"a": host_in.tolist()}, n)

module = cp.RawModule(path="add_one.ptx")
fn = module.get_function("add_one")
data = cp.asarray(host_in)
out = cp.full(n, 0x7EADBEEF, dtype=cp.int32)
fn((1,), (32,), (data, out, np.uint32(n)))
cp.cuda.Device().synchronize()

gpu = out.get().tolist()
print("input", host_in.tolist())
print("gpu   ", gpu)
print("ref   ", ref["c"])
print("MATCH" if gpu == ref["c"] == [1, 11, 21, 31, 41, 51, 61, 71] else "MISMATCH")

input [0, 10, 20, 30, 40, 50, 60, 70]
gpu    [1, 11, 21, 31, 41, 51, 61, 71]
ref    [1, 11, 21, 31, 41, 51, 61, 71]
MATCH


In [5]:
from pathlib import Path
import numpy as np
import cupy as cp

from minikernel.frontend import compile_file
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.interpreter import interpret

kernel = compile_file("examples/safe_div.mk")
Path("safe_div.ptx").write_text(compile_to_ptx(kernel))

a = np.array([7, -7, -2147483648, 5, 0], dtype=np.int32)
b = np.array([2, 0, -1, 0, 3], dtype=np.int32)
n = len(a)
expected = interpret(kernel, {"a": a.tolist(), "b": b.tolist()}, n)["c"]

module = cp.RawModule(path="safe_div.ptx")
fn = module.get_function("safe_div")
d_a, d_b = cp.asarray(a), cp.asarray(b)
d_c = cp.full(n, 0x7EADBEEF, dtype=cp.int32)
fn((1,), (32,), (d_a, d_b, d_c, np.uint32(n)))
cp.cuda.Device().synchronize()

gpu = d_c.get().tolist()
print("a       ", a.tolist())
print("b       ", b.tolist())
print("gpu     ", gpu)
print("expected", expected)
print("MATCH" if gpu == expected else "MISMATCH")

a        [7, -7, -2147483648, 5, 0]
b        [2, 0, -1, 0, 3]
gpu      [3, 0, -2147483648, 0, 0]
expected [3, 0, -2147483648, 0, 0]
MATCH


In [6]:
from pathlib import Path
import numpy as np
import cupy as cp

from minikernel.frontend import compile_file
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.interpreter import interpret

kernel = compile_file("examples/example.mk")
Path("example.ptx").write_text(compile_to_ptx(kernel))

n = 8
a = np.array([0, 1, 50, -3, 2147483647, -2147483648, 7, 100], dtype=np.int32)
b = np.array([7, 0, 9, -1, 7, 3, 8, 7], dtype=np.int32)
expected = interpret(kernel, {"a": a.tolist(), "b": b.tolist()}, n)["c"]

module = cp.RawModule(path="example.ptx")
fn = module.get_function("example")
d_a, d_b = cp.asarray(a), cp.asarray(b)
d_c = cp.full(n, 0x7EADBEEF, dtype=cp.int32)
fn((1,), (32,), (d_a, d_b, d_c, np.uint32(n)))
cp.cuda.Device().synchronize()

gpu = d_c.get().tolist()
print("gpu     ", gpu)
print("expected", expected)
print("MATCH" if gpu == expected else "MISMATCH")

gpu      [0, -36, 284, 33, 268435469, 268435488, -38, -80]
expected [0, -36, 284, 33, 268435469, 268435488, -38, -80]
MATCH


In [7]:
from pathlib import Path
import numpy as np
import cupy as cp

from minikernel.checker import check
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.generate import generate_inputs, generate_kernel
from minikernel.interpreter import interpret

def run_ptx(ptx, name, inputs, outputs, n):
    path = Path(f"{name}.ptx")
    path.write_text(ptx)
    fn = cp.RawModule(path=str(path)).get_function(name)
    dev_in = [cp.asarray(np.array(inputs[k], dtype=np.int32)) for k in inputs]
    dev_out = [cp.full(n, 0x7EADBEEF, dtype=cp.int32) for _ in outputs]
    fn((1,), (32,), (*dev_in, *dev_out, np.uint32(n)))
    cp.cuda.Device().synchronize()
    return {name: buf.get().tolist() for name, buf in zip(outputs, dev_out)}

n = 8
passed = failed = 0
for seed in range(10):
    kernel = generate_kernel(seed, depth=2)
    check(kernel)
    inputs = generate_inputs(seed, n, kernel.inputs)
    ref = interpret(kernel, inputs, n)
    try:
        gpu = run_ptx(compile_to_ptx(kernel), kernel.name, inputs, kernel.outputs, n)
    except Exception as exc:
        failed += 1
        print(f"seed {seed}: TOOL ERROR {exc}")
        continue
    if gpu == ref:
        passed += 1
        print(f"seed {seed}: PASS")
    else:
        failed += 1
        print(f"seed {seed}: MISMATCH")
        print("  gpu", gpu)
        print("  ref", ref)

print(f"\n{passed}/10 PASS, {failed} failed")


seed 0: PASS
seed 1: PASS
seed 2: PASS
seed 3: PASS
seed 4: PASS
seed 5: PASS
seed 6: PASS
seed 7: PASS
seed 8: PASS
seed 9: PASS

10/10 PASS, 0 failed


In [8]:
from pathlib import Path
from minikernel.frontend import compile_file
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.generate import generate_kernel

kernels = [
    ("add_one", compile_file("examples/add_one.mk")),
    ("safe_div", compile_file("examples/safe_div.mk")),
    ("example", compile_file("examples/example.mk")),
]
for seed in range(5):
    kernels.append((f"k{seed}", generate_kernel(seed)))

ok = 0
for name, kernel in kernels:
    path = Path(f"{name}.ptx")
    path.write_text(compile_to_ptx(kernel))
    # 2>&1 must be inside the ! line, not in Python
    out = !ptxas -arch=sm_75 {path} -o {name}.cubin 2>&1
    text = "\n".join(out)
    if "error" in text.lower() or "fatal" in text.lower():
        print(f"{name}: FAIL")
        print(text)
    else:
        ok += 1
        print(f"{name}: ptxas OK")

print(f"\n{ok}/{len(kernels)} accepted by ptxas")

add_one: ptxas OK
safe_div: ptxas OK
example: ptxas OK
k0: ptxas OK
k1: ptxas OK
k2: ptxas OK
k3: ptxas OK
k4: ptxas OK

8/8 accepted by ptxas
